In [389]:
import pandas as pd
import os
import re
import numpy as np

In [390]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input/whirlpool")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output/whirlpool")

In [391]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    df = df.copy()

    # Extrae el sector: lo que está entre TX: y el siguiente :
    # "Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ" -> "AGRC"
    df["sector"] = df[strategy_col].str.extract(r"TX:([A-Z]{3,6}):", expand=False)

    # Caso especial baseline
    df.loc[df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # Extrae transformation_name: lo que está después de TX:SECTOR:
    # "Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ" -> "DEC_CH4_RICE_STRATEGY_NZ from NZ"
    df["transformation_name"] = df[strategy_col].str.extract(r"TX:[A-Z]{3,6}:(\S+)", expand=False)

    # Si quieres solo hasta el espacio (sin "from NZ"), usa \S+ que ya captura hasta el primer espacio
    # Si quieres todo lo que sigue incluyendo "from NZ", cambia \S+ por (.+)

    # Caso baseline
    base_mask = df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [392]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "raw_emissions_uganda_2019_whirlpool_data_raw.csv"))
# emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_bulgaria_2022_trww_debug.csv"))
emissions_df.head()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.241921,2019,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
1,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.244762,2020,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
2,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252282,2021,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
3,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.251119,2022,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE
4,0.0,0.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252083,2023,ch4,0.0,0.0,Strategy TX:BASE,UGA,uganda,SISEPUEDE


In [393]:
print(emissions_df.primary_id.nunique())

64


In [394]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'NZ',
       'Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ from NZ',
       'Remove TX:CCSQ:INC_CAPTURE_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ from NZ',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_NZ from NZ',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_NZ from NZ',
       'Remove TX:FRST:INCREASE_SEQUESTRATION_NZ from NZ',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ from NZ',
       'Remove TX:INEN:INC_EFFICIENCY_PRODUCTION_STRATEGY_NZ from NZ',
       'Remove TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ from NZ',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_NZ from NZ',
       'Remove TX:IPPU:DEC_DEMAN

In [395]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

65
63


In [396]:
filtered_emissions_df["strategy"].unique()

array(['NZ', 'Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ from NZ',
       'Remove TX:CCSQ:INC_CAPTURE_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ from NZ',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_NZ from NZ',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_NZ from NZ',
       'Remove TX:FRST:INCREASE_SEQUESTRATION_NZ from NZ',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ from NZ',
       'Remove TX:INEN:INC_EFFICIENCY_PRODUCTION_STRATEGY_NZ from NZ',
       'Remove TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ from NZ',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_NZ from NZ',
       'Remove TX:IPPU:DEC_DEMAND_STRATEGY_NZ from NZ',
   

In [397]:
filtered_emissions_df.head()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
2808,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.241921,2019,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE
2809,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.244762,2020,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE
2810,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252282,2021,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE
2811,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.251119,2022,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE
2812,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252083,2023,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE


In [398]:
filtered_emissions_df.tail()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
179707,6561.0,137137.0,Wetlands:co2,Wetlands,LULUCF,0.0,2066,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from NZ,UGA,uganda,SISEPUEDE
179708,6561.0,137137.0,Wetlands:co2,Wetlands,LULUCF,0.0,2067,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from NZ,UGA,uganda,SISEPUEDE
179709,6561.0,137137.0,Wetlands:co2,Wetlands,LULUCF,0.0,2068,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from NZ,UGA,uganda,SISEPUEDE
179710,6561.0,137137.0,Wetlands:co2,Wetlands,LULUCF,0.0,2069,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from NZ,UGA,uganda,SISEPUEDE
179711,6561.0,137137.0,Wetlands:co2,Wetlands,LULUCF,0.0,2070,co2,0.0,0.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from NZ,UGA,uganda,SISEPUEDE


In [399]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['NZ', 'Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ from NZ',
       'Remove TX:CCSQ:INC_CAPTURE_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ from NZ',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_NZ from NZ',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_NZ from NZ',
       'Remove TX:FRST:INCREASE_SEQUESTRATION_NZ from NZ',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ from NZ',
       'Remove TX:INEN:INC_EFFICIENCY_PRODUCTION_STRATEGY_NZ from NZ',
       'Remove TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ from NZ',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_NZ from NZ',
       'Remove TX:IPPU:DEC_DEMAND_STRATEGY_NZ from NZ',
   

In [400]:
tornado_emissions_df.head()

,strategy_id,primary_id,Subsector_Category,CSC.Subsector,CSC.Sector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
2808,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.241921,2019,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE
2809,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.244762,2020,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE
2810,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252282,2021,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE
2811,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.251119,2022,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE
2812,6004.0,70070.0,Agriculture and Managed Soil:ch4,Agriculture and Managed Soil,Non-LULUCF,0.252083,2023,ch4,0.0,0.0,NZ,UGA,uganda,SISEPUEDE


In [401]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,value
0,6004.0,70070.0,NZ,4550.506146
1,6500.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,4896.991126
2,6501.0,77077.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4985.845114
3,6502.0,78078.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4893.051485
4,6503.0,79079.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5174.468204


In [402]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,value
58,6557.0,133133.0,Remove TX:WASO:INC_CAPTURE_BIOGAS_STRATEGY_NZ ...,5017.234129
59,6558.0,134134.0,Remove TX:WASO:INC_ENERGY_FROM_BIOGAS_STRATEGY...,4884.446631
60,6559.0,135135.0,Remove TX:WASO:INC_ENERGY_FROM_INCINERATION_ST...,4896.598436
61,6560.0,136136.0,Remove TX:WASO:INC_LANDFILLING_STRATEGY_NZ fro...,4909.626424
62,6561.0,137137.0,Remove TX:WASO:INC_RECYCLING_STRATEGY_NZ from NZ,4919.797273


In [403]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

63
63


In [ ]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total column by setting it to the strategy_id == 0 value
base_emission_total = (tornado_emissions_agg_df.loc[tornado_emissions_agg_df['strategy_id'] == 6004, 'emission_total'].values[0]) - 400
tornado_emissions_agg_df['base_emission_total'] = base_emission_total 

base_emission_total

np.float64(4150.506146230362)

In [405]:
# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] =  tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']
tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff
0,6004.0,70070.0,NZ,4550.506146,4150.506146,400.000000
1,6500.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,4896.991126,4150.506146,746.484980
2,6501.0,77077.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4985.845114,4150.506146,835.338968
3,6502.0,78078.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4893.051485,4150.506146,742.545339
4,6503.0,79079.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5174.468204,4150.506146,1023.962057


In [406]:
tornado_emissions_agg_extended_df = add_sector_and_transformation_fields(tornado_emissions_agg_df)
tornado_emissions_agg_extended_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,6004.0,70070.0,NZ,4550.506146,4150.506146,400.000000,NaN,
1,6500.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,4896.991126,4150.506146,746.484980,AGRC,DEC_CH4_RICE_STRATEGY_NZ
2,6501.0,77077.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4985.845114,4150.506146,835.338968,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ
3,6502.0,78078.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4893.051485,4150.506146,742.545339,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ
4,6503.0,79079.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5174.468204,4150.506146,1023.962057,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ


In [407]:
tornado_emissions_agg_extended_df.to_clipboard(index=False)

## Load and process CB data

In [408]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "cba_results_ssp_modeling_whirlpool.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value
0,PFLO:NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
1,PFLO:NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
2,PFLO:NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
3,PFLO:NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
4,PFLO:NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0


In [409]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

# Split 'variable' into components: name, sector, cb_type, item_1, item_2
# (Assumes exactly 5 colon-separated parts; if there are more colons inside the last field,
# they will be kept in item_2 thanks to n=4)
cb_chars = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
cb_chars.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
cb_data = pd.concat([cb_data, cb_chars], axis=1)
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2
0,PFLO:NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
1,PFLO:NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
2,PFLO:NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
3,PFLO:NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
4,PFLO:NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural


In [410]:
# Scale value from USD to billions (divide by 1e9)
if "value" in cb_data.columns:
    cb_data["value"] = cb_data["value"] / 1e9

# --- Remove "shifted" entries ---
# # Remove rows where item_2 contains "shifted"
# cb_data = cb_data[~cb_data["item_2"].astype(str).str.contains("shifted", na=False)]

# # Remove any remaining rows where variable contains "shifted2"
# cb_data = cb_data[~cb_data["variable"].astype(str).str.contains("shifted2", na=False)]

# --- Add Year column (Year = time_period + 2015) ---
cb_data["Year"] = cb_data["time_period"] + 2015

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year
0,PFLO:NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019
1,PFLO:NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020
2,PFLO:NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021
3,PFLO:NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022
4,PFLO:NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023


In [411]:
# Load attribute strategy
attribute_strategy_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "ATTRIBUTE_STRATEGY.csv"))
attribute_strategy_df = attribute_strategy_df[["strategy_id", "strategy_code"]]
attribute_strategy_df.head()

,strategy_id,strategy_code
0,6004,PFLO:NZ
1,0,BASE
2,4004,IPPU:DEC_OTHER_FCS
3,6500,WHIRLPOOL:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ
4,6501,WHIRLPOOL:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRA...


In [412]:
attribute_strategy_df.strategy_id.unique()

array([6004,    0, 4004, 6500, 6501, 6502, 6503, 6504, 6505, 6506, 6507,
       6508, 6509, 6510, 6511, 6512, 6513, 6514, 6515, 6516, 6517, 6518,
       6519, 6520, 6521, 6522, 6523, 6524, 6525, 6526, 6527, 6528, 6529,
       6530, 6531, 6532, 6533, 6534, 6535, 6536, 6537, 6538, 6539, 6540,
       6541, 6542, 6543, 6544, 6545, 6546, 6547, 6548, 6549, 6550, 6551,
       6552, 6553, 6554, 6555, 6556, 6557, 6558, 6559, 6560, 6561])

In [413]:
attribute_strategy_df.strategy_code.unique()

array(['PFLO:NZ', 'BASE', 'IPPU:DEC_OTHER_FCS',
       'WHIRLPOOL:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ',
       'WHIRLPOOL:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ',
       'WHIRLPOOL:TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ',
       'WHIRLPOOL:TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ',
       'WHIRLPOOL:TX:CCSQ:INC_CAPTURE_STRATEGY_NZ',
       'WHIRLPOOL:TX:ENTC:DEC_LOSSES_STRATEGY_NZ',
       'WHIRLPOOL:TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ',
       'WHIRLPOOL:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ',
       'WHIRLPOOL:TX:FGTV:DEC_LEAKS_STRATEGY_NZ',
       'WHIRLPOOL:TX:FGTV:INC_FLARE_STRATEGY_NZ',
       'WHIRLPOOL:TX:FRST:INCREASE_SEQUESTRATION_NZ',
       'WHIRLPOOL:TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ',
       'WHIRLPOOL:TX:INEN:INC_EFFICIENCY_PRODUCTION_STRATEGY_NZ',
       'WHIRLPOOL:TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ',
       'WHIRLPOOL:TX:IPPU:DEC_CLINKER_STRATEGY_NZ',
       'WHIRLPOOL:TX:IPPU:DEC_DEMAND_STRATEGY_NZ',
       'WHIRLPOOL:TX:IPPU:DEC_HFCS_STRATE

In [414]:
# Merge with cb_data on strategy_code
cb_data = cb_data.merge(attribute_strategy_df, on="strategy_code", how="left")
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6004
1,PFLO:NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6004
2,PFLO:NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6004
3,PFLO:NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6004
4,PFLO:NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6004


In [415]:
cb_data.strategy_code.unique()

array(['PFLO:NZ', 'WHIRLPOOL:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ',
       'WHIRLPOOL:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ',
       'WHIRLPOOL:TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ',
       'WHIRLPOOL:TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ',
       'WHIRLPOOL:TX:CCSQ:INC_CAPTURE_STRATEGY_NZ',
       'WHIRLPOOL:TX:ENTC:DEC_LOSSES_STRATEGY_NZ',
       'WHIRLPOOL:TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ',
       'WHIRLPOOL:TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ',
       'WHIRLPOOL:TX:FGTV:DEC_LEAKS_STRATEGY_NZ',
       'WHIRLPOOL:TX:FGTV:INC_FLARE_STRATEGY_NZ',
       'WHIRLPOOL:TX:FRST:INCREASE_SEQUESTRATION_NZ',
       'WHIRLPOOL:TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ',
       'WHIRLPOOL:TX:INEN:INC_EFFICIENCY_PRODUCTION_STRATEGY_NZ',
       'WHIRLPOOL:TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ',
       'WHIRLPOOL:TX:IPPU:DEC_CLINKER_STRATEGY_NZ',
       'WHIRLPOOL:TX:IPPU:DEC_DEMAND_STRATEGY_NZ',
       'WHIRLPOOL:TX:IPPU:DEC_HFCS_STRATEGY_NZ',
       'WHIRLPOOL:TX:IPPU:DEC

In [416]:
cb_data.strategy_id.unique()

array([6004, 6500, 6501, 6502, 6503, 6504, 6505, 6506, 6507, 6508, 6509,
       6510, 6511, 6512, 6513, 6514, 6515, 6516, 6517, 6518, 6519, 6520,
       6521, 6522, 6523, 6524, 6525, 6526, 6527, 6528, 6529, 6530, 6531,
       6532, 6533, 6534, 6535, 6536, 6537, 6538, 6539, 6540, 6541, 6542,
       6543, 6544, 6545, 6546, 6547, 6548, 6549, 6550, 6551, 6552, 6553,
       6554, 6555, 6556, 6557, 6558, 6559, 6560, 6561])

In [417]:
# check for nans in strategy_id
cb_data[cb_data['strategy_id'].isna()]['strategy_code'].unique()

array([], dtype=object)

In [418]:
cb_data["sector"].unique()

array(['wali', 'entc', 'trns', 'lndu', 'waso', 'trww', 'lvst', 'agrc',
       'ccsq', 'inen', 'scoe', 'ippu', 'soil', 'lsmm', 'fgtv', 'pflo'],
      dtype=object)

In [419]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6004
1,PFLO:NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6004
2,PFLO:NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6004
3,PFLO:NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6004
4,PFLO:NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6004


In [420]:
# filter sectors
# target_sectors = ["wali", "trww", "waso", "soil", "ippu", "lvst", "agrc", "lndu", "lsmm"]
# cb_data = cb_data[cb_data["sector"].isin(target_sectors)].copy()
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:NZ,0,uganda,4,pop_unimproved_rural,1.932346e+07,1.932346e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2019,6004
1,PFLO:NZ,0,uganda,5,pop_unimproved_rural,1.951974e+07,1.951974e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2020,6004
2,PFLO:NZ,0,uganda,6,pop_unimproved_rural,1.896047e+07,1.896047e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2021,6004
3,PFLO:NZ,0,uganda,7,pop_unimproved_rural,1.837001e+07,1.837001e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022,6004
4,PFLO:NZ,0,uganda,8,pop_unimproved_rural,1.774752e+07,1.774752e+07,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023,6004


In [421]:
# aggregate sum(value) grouped by strategy_id and cb_type
cb_data = (
    cb_data.groupby(["strategy_id", "cb_type"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head(20)

,strategy_id,cb_type,cumulative
0,6004,air_pollution,22.170309
1,6004,congestion,21.232368
2,6004,consumer_savings,712.760132
3,6004,crop_value,60.840256
4,6004,ecosystem_services,21.469016
5,6004,env_pollution,101.384149
6,6004,fuel_cost,143.602466
7,6004,human_health,739.678328
8,6004,ippu_value,7.062776
9,6004,land_pollution,0.550622


In [422]:
# unique cb_data types
cb_cats = cb_data["cb_type"].unique().tolist()

# long -> wide (R dcast equivalent)
wide_cb = (
    cb_data.pivot(index="strategy_id", columns="cb_type", values="cumulative")
      .reset_index()
)

# optional: remove column name from pivot for nicer printing
wide_cb.columns.name = None
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,6004,22.170309,21.232368,712.760132,60.840256,21.469016,101.384149,143.602466,739.678328,7.062776,0.550622,-53.526719,28.985268,2.746977,282.977194,-854.515984,159.221581,235.431542
1,6500,25.900815,20.962109,707.848423,60.780520,20.806639,99.868380,141.277637,733.968321,6.930198,0.539352,-52.897205,28.616772,28.077653,279.293719,-844.980814,157.009056,234.643933
2,6501,25.893883,20.962109,767.352760,126.047796,18.800549,99.008350,145.420852,733.968321,6.930198,0.493093,-54.008302,28.616772,28.070639,279.293719,-866.491079,3.347959,234.643933
3,6502,25.900815,20.962109,668.877670,60.780520,20.806639,99.868380,141.277637,733.968321,6.930198,0.539352,-52.897205,28.616772,28.077653,279.293719,-845.133798,152.443910,234.643933
4,6503,25.950885,20.962109,720.044440,-69.797847,9.815789,99.868380,140.437081,733.968321,6.930198,0.272580,-65.003887,28.616772,28.627720,279.293719,-841.233764,158.441387,234.643931


In [423]:
cb_cats

['air_pollution',
 'congestion',
 'consumer_savings',
 'crop_value',
 'ecosystem_services',
 'env_pollution',
 'fuel_cost',
 'human_health',
 'ippu_value',
 'land_pollution',
 'lvst_value',
 'road_safety',
 'sector_specific',
 'system_cost',
 'technical_cost',
 'technical_savings',
 'water_pollution']

In [424]:
# --- 1) net_benefit = rowSums over all cb categories ---
wide_cb["net_benefit"] = wide_cb[cb_cats].sum(axis=1, skipna=True)

# --- 2) additional_benefits = rowSums excluding "technical_cost" ---
benefit_cols = [c for c in cb_cats if c != "technical_cost"]
wide_cb["additional_benefits"] = wide_cb[benefit_cols].sum(axis=1, skipna=True)

# --- 3) total_transformation_costs = rowSums over specific cols ---
cost_cols = ["technical_cost", "technical_savings", "fuel_cost"]

# (safe version: only use cols that exist in the df)
cost_cols = [c for c in cost_cols if c in wide_cb.columns]

wide_cb["total_transformation_costs"] = wide_cb[cost_cols].sum(axis=1, skipna=True)
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6004,22.170309,21.232368,712.760132,60.840256,21.469016,101.384149,143.602466,739.678328,7.062776,...,-53.526719,28.985268,2.746977,282.977194,-854.515984,159.221581,235.431542,1632.070280,2486.586265,-551.691937
1,6500,25.900815,20.962109,707.848423,60.780520,20.806639,99.868380,141.277637,733.968321,6.930198,...,-52.897205,28.616772,28.077653,279.293719,-844.980814,157.009056,234.643933,1648.645507,2493.626321,-546.694121
2,6501,25.893883,20.962109,767.352760,126.047796,18.800549,99.008350,145.420852,733.968321,6.930198,...,-54.008302,28.616772,28.070639,279.293719,-866.491079,3.347959,234.643933,1598.351552,2464.842630,-717.722267
3,6502,25.900815,20.962109,668.877670,60.780520,20.806639,99.868380,141.277637,733.968321,6.930198,...,-52.897205,28.616772,28.077653,279.293719,-845.133798,152.443910,234.643933,1604.956625,2450.090423,-551.412250
4,6503,25.950885,20.962109,720.044440,-69.797847,9.815789,99.868380,140.437081,733.968321,6.930198,...,-65.003887,28.616772,28.627720,279.293719,-841.233764,158.441387,234.643931,1511.837812,2353.071576,-542.355297


## Merge emissions and cb data and save

In [425]:
tornado_emissions_agg_extended_df[tornado_emissions_agg_extended_df.strategy == "NZ"]

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,6004.0,70070.0,NZ,4550.506146,4150.506146,400.0,NaN,


In [426]:
wide_cb.strategy_id.unique()

array([6004, 6500, 6501, 6502, 6503, 6504, 6505, 6506, 6507, 6508, 6509,
       6510, 6511, 6512, 6513, 6514, 6515, 6516, 6517, 6518, 6519, 6520,
       6521, 6522, 6523, 6524, 6525, 6526, 6527, 6528, 6529, 6530, 6531,
       6532, 6533, 6534, 6535, 6536, 6537, 6538, 6539, 6540, 6541, 6542,
       6543, 6544, 6545, 6546, 6547, 6548, 6549, 6550, 6551, 6552, 6553,
       6554, 6555, 6556, 6557, 6558, 6559, 6560, 6561])

In [427]:
print(wide_cb.shape)
print(tornado_emissions_agg_extended_df.shape)

(63, 21)
(63, 8)


In [428]:
df_merged = pd.merge(
    tornado_emissions_agg_extended_df,
    wide_cb,
    on="strategy_id",
    how="inner"
)

df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6004.0,70070.0,NZ,4550.506146,4150.506146,400.000000,NaN,,22.170309,21.232368,...,-53.526719,28.985268,2.746977,282.977194,-854.515984,159.221581,235.431542,1632.070280,2486.586265,-551.691937
1,6500.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,4896.991126,4150.506146,746.484980,AGRC,DEC_CH4_RICE_STRATEGY_NZ,25.900815,20.962109,...,-52.897205,28.616772,28.077653,279.293719,-844.980814,157.009056,234.643933,1648.645507,2493.626321,-546.694121
2,6501.0,77077.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4985.845114,4150.506146,835.338968,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,25.893883,20.962109,...,-54.008302,28.616772,28.070639,279.293719,-866.491079,3.347959,234.643933,1598.351552,2464.842630,-717.722267
3,6502.0,78078.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4893.051485,4150.506146,742.545339,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,25.900815,20.962109,...,-52.897205,28.616772,28.077653,279.293719,-845.133798,152.443910,234.643933,1604.956625,2450.090423,-551.412250
4,6503.0,79079.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5174.468204,4150.506146,1023.962057,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ,25.950885,20.962109,...,-65.003887,28.616772,28.627720,279.293719,-841.233764,158.441387,234.643931,1511.837812,2353.071576,-542.355297


In [429]:
df_merged = df_merged[df_merged['strategy'] != 'NZ']

df_merged.head()


,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
1,6500.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,4896.991126,4150.506146,746.484980,AGRC,DEC_CH4_RICE_STRATEGY_NZ,25.900815,20.962109,...,-52.897205,28.616772,28.077653,279.293719,-844.980814,157.009056,234.643933,1648.645507,2493.626321,-546.694121
2,6501.0,77077.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4985.845114,4150.506146,835.338968,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,25.893883,20.962109,...,-54.008302,28.616772,28.070639,279.293719,-866.491079,3.347959,234.643933,1598.351552,2464.842630,-717.722267
3,6502.0,78078.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4893.051485,4150.506146,742.545339,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,25.900815,20.962109,...,-52.897205,28.616772,28.077653,279.293719,-845.133798,152.443910,234.643933,1604.956625,2450.090423,-551.412250
4,6503.0,79079.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5174.468204,4150.506146,1023.962057,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ,25.950885,20.962109,...,-65.003887,28.616772,28.627720,279.293719,-841.233764,158.441387,234.643931,1511.837812,2353.071576,-542.355297
5,6504.0,80080.0,Remove TX:CCSQ:INC_CAPTURE_STRATEGY_NZ from NZ,4901.352404,4150.506146,750.846257,CCSQ,INC_CAPTURE_STRATEGY_NZ,25.900827,20.962109,...,-52.897205,28.616772,28.742422,279.293719,-843.922124,157.009056,234.643933,1650.368977,2494.291101,-545.635431


In [430]:
print(df_merged.shape)

(62, 28)


### Below we have some hardcoded fixed exclusive of this study case to replace incorrect tranformation names

In [431]:
df_merged.columns

Index(['strategy_id', 'primary_id', 'strategy', 'emission_total',
       'base_emission_total', 'emission_diff', 'sector', 'transformation_name',
       'air_pollution', 'congestion', 'consumer_savings', 'crop_value',
       'ecosystem_services', 'env_pollution', 'fuel_cost', 'human_health',
       'ippu_value', 'land_pollution', 'lvst_value', 'road_safety',
       'sector_specific', 'system_cost', 'technical_cost', 'technical_savings',
       'water_pollution', 'net_benefit', 'additional_benefits',
       'total_transformation_costs'],
      dtype='object')

In [432]:
# Get the base technical cost from wide_cb (strategy_id 6004)
base_technical_cost = (wide_cb[wide_cb['strategy_id'] == 6004]['technical_cost'].values[0])*-1
base_technical_cost

np.float64(854.5159842852808)

In [433]:
# multiply technical_cost by -1 to get positive costs
df_merged['technical_cost'] = df_merged['technical_cost'] * -1

df_merged['technical_cost'] = base_technical_cost - df_merged['technical_cost'] 

# create marginal total abatement cost column
df_merged['marginal_total_abatement_cost_(USD/tCO2e)'] = (df_merged['technical_cost'] / df_merged['emission_diff'])*1000

# If technical_cost is positive then marginal_total_abatement_cost should be positive too.
df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs() * np.sign(df_merged["technical_cost"])


In [434]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e)
1,6500.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,4896.991126,4150.506146,746.484980,AGRC,DEC_CH4_RICE_STRATEGY_NZ,25.900815,20.962109,...,28.616772,28.077653,279.293719,9.535170,157.009056,234.643933,1648.645507,2493.626321,-546.694121,12.773425
2,6501.0,77077.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4985.845114,4150.506146,835.338968,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,25.893883,20.962109,...,28.616772,28.070639,279.293719,-11.975094,3.347959,234.643933,1598.351552,2464.842630,-717.722267,-14.335611
3,6502.0,78078.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4893.051485,4150.506146,742.545339,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,25.900815,20.962109,...,28.616772,28.077653,279.293719,9.382187,152.443910,234.643933,1604.956625,2450.090423,-551.412250,12.635170
4,6503.0,79079.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5174.468204,4150.506146,1023.962057,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ,25.950885,20.962109,...,28.616772,28.627720,279.293719,13.282220,158.441387,234.643931,1511.837812,2353.071576,-542.355297,12.971399
5,6504.0,80080.0,Remove TX:CCSQ:INC_CAPTURE_STRATEGY_NZ from NZ,4901.352404,4150.506146,750.846257,CCSQ,INC_CAPTURE_STRATEGY_NZ,25.900827,20.962109,...,28.616772,28.742422,279.293719,10.593860,157.009056,234.643933,1650.368977,2494.291101,-545.635431,14.109226


In [435]:
df_merged["strategy"].unique()

array(['Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ from NZ',
       'Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ from NZ',
       'Remove TX:CCSQ:INC_CAPTURE_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:DEC_LOSSES_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN_STRATEGY_NZ from NZ',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC_STRATEGY_NZ from NZ',
       'Remove TX:FGTV:DEC_LEAKS_STRATEGY_NZ from NZ',
       'Remove TX:FGTV:INC_FLARE_STRATEGY_NZ from NZ',
       'Remove TX:FRST:INCREASE_SEQUESTRATION_NZ from NZ',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY_STRATEGY_NZ from NZ',
       'Remove TX:INEN:INC_EFFICIENCY_PRODUCTION_STRATEGY_NZ from NZ',
       'Remove TX:INEN:SHIFT_FUEL_HEAT_STRATEGY_NZ from NZ',
       'Remove TX:IPPU:DEC_CLINKER_STRATEGY_NZ from NZ',
       'Remove TX:IPPU:DEC_DEMAND_STRATEGY_NZ from NZ',
       'R

In [436]:
df_merged.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_whirlpool.csv"), index=False)

### Create a QA version

In [437]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e)
1,6500.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,4896.991126,4150.506146,746.484980,AGRC,DEC_CH4_RICE_STRATEGY_NZ,25.900815,20.962109,...,28.616772,28.077653,279.293719,9.535170,157.009056,234.643933,1648.645507,2493.626321,-546.694121,12.773425
2,6501.0,77077.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN_STRATEG...,4985.845114,4150.506146,835.338968,AGRC,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,25.893883,20.962109,...,28.616772,28.070639,279.293719,-11.975094,3.347959,234.643933,1598.351552,2464.842630,-717.722267,-14.335611
3,6502.0,78078.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE_ST...,4893.051485,4150.506146,742.545339,AGRC,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,25.900815,20.962109,...,28.616772,28.077653,279.293719,9.382187,152.443910,234.643933,1604.956625,2450.090423,-551.412250,12.635170
4,6503.0,79079.0,Remove TX:AGRC:INC_PRODUCTIVITY_STRATEGY_NZ fr...,5174.468204,4150.506146,1023.962057,AGRC,INC_PRODUCTIVITY_STRATEGY_NZ,25.950885,20.962109,...,28.616772,28.627720,279.293719,13.282220,158.441387,234.643931,1511.837812,2353.071576,-542.355297,12.971399
5,6504.0,80080.0,Remove TX:CCSQ:INC_CAPTURE_STRATEGY_NZ from NZ,4901.352404,4150.506146,750.846257,CCSQ,INC_CAPTURE_STRATEGY_NZ,25.900827,20.962109,...,28.616772,28.742422,279.293719,10.593860,157.009056,234.643933,1650.368977,2494.291101,-545.635431,14.109226


In [438]:
df_merged.sector.unique()

array(['AGRC', 'CCSQ', 'ENTC', 'FGTV', 'FRST', 'INEN', 'IPPU', 'LNDU',
       'LSMM', 'LVST', 'PFLO', 'SCOE', 'SOIL', 'TRDE', 'TRNS', 'TRWW',
       'WALI', 'WASO'], dtype=object)

In [439]:
relevant_fields = [
    "transformation_name",
    "sector",
    "base_emission_total",
    "emission_total",
    "emission_diff",
    "technical_cost",
    "marginal_total_abatement_cost_(USD/tCO2e)"
]

# keep only relevant fields
df_merged_filtered = df_merged[relevant_fields]
df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,DEC_CH4_RICE_STRATEGY_NZ,AGRC,4150.506146,4896.991126,746.484980,9.535170,12.773425
2,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,AGRC,4150.506146,4985.845114,835.338968,-11.975094,-14.335611
3,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,AGRC,4150.506146,4893.051485,742.545339,9.382187,12.635170
4,INC_PRODUCTIVITY_STRATEGY_NZ,AGRC,4150.506146,5174.468204,1023.962057,13.282220,12.971399
5,INC_CAPTURE_STRATEGY_NZ,CCSQ,4150.506146,4901.352404,750.846257,10.593860,14.109226


In [440]:
df_merged_filtered

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,DEC_CH4_RICE_STRATEGY_NZ,AGRC,4150.506146,4896.991126,746.484980,9.535170,12.773425
2,DEC_LOSSES_SUPPLY_CHAIN_STRATEGY_NZ,AGRC,4150.506146,4985.845114,835.338968,-11.975094,-14.335611
3,INC_CONSERVATION_AGRICULTURE_STRATEGY_NZ,AGRC,4150.506146,4893.051485,742.545339,9.382187,12.635170
4,INC_PRODUCTIVITY_STRATEGY_NZ,AGRC,4150.506146,5174.468204,1023.962057,13.282220,12.971399
5,INC_CAPTURE_STRATEGY_NZ,CCSQ,4150.506146,4901.352404,750.846257,10.593860,14.109226
...,...,...,...,...,...,...,...
58,INC_CAPTURE_BIOGAS_STRATEGY_NZ,WASO,4150.506146,5017.234129,866.727982,59.174003,68.272866
59,INC_ENERGY_FROM_BIOGAS_STRATEGY_NZ,WASO,4150.506146,4884.446631,733.940484,29.435384,40.105955
60,INC_ENERGY_FROM_INCINERATION_STRATEGY_NZ,WASO,4150.506146,4896.598436,746.092289,7.916103,10.610085
61,INC_LANDFILLING_STRATEGY_NZ,WASO,4150.506146,4909.626424,759.120278,68.391723,90.093395


In [441]:
df_merged_filtered.to_clipboard(index=False)

In [442]:
df_merged_filtered.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_for_QA_whirlpool.csv"), index=False)